# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process a scientific dataset described by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset Name:', metadata.name)
print('Description:', metadata.description)
print('Cite as:', getattr(metadata, 'citeAs', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The dataset may contain several `recordSet` objects, each defining a table or file-like structure.

We'll enumerate all record sets, listing their `@id`, name, fields, and available columns.

In [ ]:
# List all record sets and their fields
if not metadata.recordSet:
    print('No recordSets defined in the top-level schema. Trying to auto-enumerate from schema...')
    # Fallback: force-load one record set to inspect structure
record_set_ids = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        print(f"RecordSet: {rs['@id']}")
        record_set_ids.append(rs['@id'])
        if 'field' in rs and rs['field']:
            print('  Fields:')
            for field in rs['field']:
                print(f"   - {field['@id']} | name: {field.get('name', '')} | type: {field.get('dataType', '')}")
else:
    # Try to infer some recordSet ids from the schema
    # The proper use of Croissant expects IDs structured as URIs, but we'll try a common default
    # mlcroissant can sometimes let you enumerate available record sets if you call records() with no arguments
    # Let's try this as a guess:
    try:
        from pprint import pprint
        print('Available recordset samples:')
        sample_records = list(dataset.records())
        if sample_records:
            pprint(sample_records[0])
        else:
            print('No records found using default record set.')
    except Exception as e:
        print('Could not enumerate record sets:', e)

# Typical Croissant datasets will expose recordSet @id(s) in metadata.
# If your list above is empty, consult the schema or dataset docs for available table/file names.

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We must reference record set(s) by their `@id`. For this dataset, let's use its primary data table if available.

_If you know the specific record set `@id`, set it below. For example: `record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034#EVProteins'`._

In [ ]:
# If you discovered record set @id(s) above, add them here:

record_set_ids = [
    # Replace with actual @id(s) for record sets (e.g. from metadata.recordSet or known dataset structure)
    'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034#EVProteins',
    # Add more ids if more record sets exist
]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for recordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records.")
        print(f"Columns for {record_set_id}: {list(dataframes[record_set_id].columns)}\n")
    else:
        print(f"No records found for {record_set_id}")
# Set primary record set id for next steps
main_record_set_id = record_set_ids[0]

## 4. Exploratory Data Analysis (EDA)
Apply basic data transformations: filtering, normalizing, grouping. All fields must be referenced using their Croissant `@id`.

Let's assume (based on common proteomic tables) that there's a numeric field for, e.g., 'peptide_counts' or 'Abundance'. List the available columns to pick an appropriate field.

In [ ]:
# List columns to choose fields for EDA
print("Available columns in main DataFrame:")
print(dataframes[main_record_set_id].columns.tolist())

In [ ]:
# ---
# Set the field IDs to reference for EDA. Update these to match your dataset field IDs.
# Example IDs: 'peptide_counts', 'normalized_abundance', etc. All should be column names as loaded.

# Choose a likely-numeric field for demonstration
numeric_field = 'cr:peptide_counts'  # Example; change to your field's ID if different
group_field = 'cr:description'       # Example for grouping; or use e.g. 'cr:protein_class' if available

df = dataframes[main_record_set_id]

# Remove outliers: simple filter for this numeric field
if numeric_field in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (total: {len(filtered_df)} rows):")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records (first 5):")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field and compute means
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped mean {numeric_field} by '{group_field}' (first 5 groups):")
        print(grouped_df.head())
else:
    print(f"Numeric field '{numeric_field}' was not found! Update 'numeric_field' to match your dataset.")

## 5. Visualization
Visualize distributions or relationships between chosen fields. For example, plot histogram of the normalized field, or bar plot of group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} (> {threshold})")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(6,4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"], bins=30, kde=True, color='orange')
        plt.title(f"Normalized {numeric_field} distribution (> {threshold})")
        plt.xlabel(f"{numeric_field} normalized")
        plt.ylabel("Count")
        plt.show()

    if group_field in filtered_df.columns:
        top_n = 10
        group_means = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)[:top_n]
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_means.values, y=group_means.index, orient='h')
        plt.title(f"Top {top_n} {group_field} by mean {numeric_field}")
        plt.xlabel(f"Mean {numeric_field}")
        plt.ylabel(group_field)
        plt.tight_layout()
        plt.show()
else:
    print(f"Cannot plot: field '{numeric_field}' missing. Adjust field names above to your dataset.")

## 6. Conclusion
In this notebook, we loaded FAIR^2-formatted mass spectrometry data from a Croissant schema using mlcroissant, explored available record sets and fields via their `@id`, performed data extraction, filtering, normalization, simple grouping, and visualized important distributions.

For further exploration, refer to the full Croissant schema to access more record sets, and consult the dataset documentation for biological and clinical context. All field and record set references should use the official schema's `@id` so analyses are robust to changes or upgrades in the dataset.
